In [2]:
# VERIFICATION CELL — 24 Aug 2026. Four read-only checks before Notebook 05.
# Nothing is saved or changed; this cell only prints evidence.

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PROC = Path("..").resolve() / "data/processed"

# Load audited data and rebuild the 70/15/15 split exactly as notebook 04 does.
df = pd.read_parquet(DATA_PROC / "01_audit.parquet")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
t70 = df["Timestamp"].quantile(0.70)
t85 = df["Timestamp"].quantile(0.85)
df["split"] = np.where(df["Timestamp"] <= t70, "train",
              np.where(df["Timestamp"] <= t85, "val", "test"))
assert df["split"].value_counts().to_dict() == {
    "train": 3554957, "val": 761749, "test": 761639}, "Split mismatch"
print("Split reproduced OK\n")

# ---------- CHECK A: wind-down rule (Option A) verified by execution ----------
# Median daily volume across complete calendar days inside the training window (1-6 Sep);
# any day below 5% of that median is a wind-down day, excluded from evaluation.
df["day"] = df["Timestamp"].dt.date
daily = df.groupby("day").size()
train_days = [d for d in daily.index if pd.Timestamp(d) >= pd.Timestamp("2022-09-01")
              and pd.Timestamp(d) <= pd.Timestamp("2022-09-06")]
M = daily.loc[train_days].median()
threshold = 0.05 * M
winddown_days = set(daily[daily < threshold].index)
print(f"A) Median training-day volume M = {M:,.1f}  (expect 482,369.5)")
print(f"   Threshold 0.05*M = {threshold:,.1f}")
print(f"   Wind-down days: {sorted(winddown_days)}")
test = df[df["split"] == "test"]
test_main = test[~test["day"].isin(winddown_days)]
wd = test[test["day"].isin(winddown_days)]
print(f"   Test-main: n={len(test_main):,}  illicit={test_main['Is Laundering'].sum():,}"
      f"  (expect 760,531 / 906)")
print(f"   Wind-down: n={len(wd):,}  illicit={wd['Is Laundering'].sum():,}"
      f"  (expect 1,108 / 655)\n")

# ---------- CHECK B: are laundering rows almost always same-currency? ----------
for label, name in [(1, "Illicit"), (0, "Legitimate")]:
    sub = df[df["Is Laundering"] == label]
    same_ccy = (sub["Payment Currency"] == sub["Receiving Currency"]).mean()
    same_amt = (sub["Amount Paid"] == sub["Amount Received"]).mean()
    print(f"B) {name}: same currency {same_ccy:.2%} | identical amounts {same_amt:.2%}")
print()

# ---------- CHECK C: does any account ID appear under more than one bank? ----------
# Notebook 04 uses the account string alone as the vertex name. If one ID maps to
# several banks, the graph silently merged distinct accounts.
send = df[["From Bank", "Account"]].rename(columns={"From Bank": "bank", "Account": "acct"})
recv = df[["To Bank", "Account.1"]].rename(columns={"To Bank": "bank", "Account.1": "acct"})
pairs = pd.concat([send, recv], ignore_index=True)
pairs["acct"] = pairs["acct"].astype(str)
pairs = pairs.drop_duplicates()
banks_per_acct = pairs.groupby("acct")["bank"].nunique()
n_collisions = int((banks_per_acct > 1).sum())
print(f"C) Unique account IDs (whole dataset): {banks_per_acct.shape[0]:,}")
print(f"   IDs appearing under >1 bank: {n_collisions:,}"
      f"  ({'GRAPH IS SAFE' if n_collisions == 0 else 'PROBLEM - accounts merged'})\n")

# ---------- CHECK D: unseen accounts + staleness strata on test-main ----------
# Accounts with no training-window activity get imputed profiles (U3 Option A).
# Count them, then size the three strata against the degeneracy rule
# (any stratum < 1,000 transactions or < 10 illicit -> counts only).
tr = df[df["split"] == "train"]
train_accts = set(tr["Account"].astype(str)) | set(tr["Account.1"].astype(str))
print(f"D) Accounts in training window: {len(train_accts):,}  (expect 513,284)")

tm = test_main.copy()
seen_s = tm["Account"].astype(str).isin(train_accts)
seen_r = tm["Account.1"].astype(str).isin(train_accts)
tm["stratum"] = np.select(
    [seen_s & seen_r, seen_s ^ seen_r], ["both-active", "one-imputed"], "both-imputed")
strata = tm.groupby("stratum").agg(
    n=("Is Laundering", "size"), illicit=("Is Laundering", "sum"))
strata["degenerate"] = (strata["n"] < 1000) | (strata["illicit"] < 10)
print(strata, "\n")

all_accts = set(df["Account"].astype(str)) | set(df["Account.1"].astype(str))
unseen = len(all_accts) - len(train_accts)
print(f"   Total accounts anywhere: {len(all_accts):,}  ->  unseen in training: "
      f"{unseen:,} ({unseen/len(all_accts):.2%})")

Split reproduced OK

A) Median training-day volume M = 482,369.5  (expect 482,369.5)
   Threshold 0.05*M = 24,118.5
   Wind-down days: [datetime.date(2022, 9, 11), datetime.date(2022, 9, 12), datetime.date(2022, 9, 13), datetime.date(2022, 9, 14), datetime.date(2022, 9, 15), datetime.date(2022, 9, 16), datetime.date(2022, 9, 17), datetime.date(2022, 9, 18)]
   Test-main: n=760,531  illicit=906  (expect 760,531 / 906)
   Wind-down: n=1,108  illicit=655  (expect 1,108 / 655)

B) Illicit: same currency 100.00% | identical amounts 100.00%
B) Legitimate: same currency 98.58% | identical amounts 98.58%

C) Unique account IDs (whole dataset): 515,080
   IDs appearing under >1 bank: 8  (PROBLEM - accounts merged)

D) Accounts in training window: 513,284  (expect 513,284)
                   n  illicit  degenerate
stratum                                  
both-active   758982      878       False
both-imputed      58        0        True
one-imputed     1491       28       False 

   Total accou

In [3]:
# Cross-check for Check B: build a table of Payment Currency (rows)
# vs Receiving Currency (columns) for illicit transactions only.
# If every illicit transaction is same-currency, all counts sit on
# the diagonal and every off-diagonal cell is zero.

illicit = df[df["Is Laundering"] == 1]

ccy_table = pd.crosstab(illicit["Payment Currency"], illicit["Receiving Currency"])
print(ccy_table)
print()

# Count how many illicit rows fall OFF the diagonal (should be 0)
off_diagonal = (illicit["Payment Currency"] != illicit["Receiving Currency"]).sum()
print(f"Illicit rows where the two currencies differ: {off_diagonal}")

# And for contrast: the same count for legitimate rows (should be ~72,000)
legit = df[df["Is Laundering"] == 0]
off_diagonal_legit = (legit["Payment Currency"] != legit["Receiving Currency"]).sum()
print(f"Legitimate rows where the two currencies differ: {off_diagonal_legit:,}")

Receiving Currency  Australian Dollar  Bitcoin  Brazil Real  Canadian Dollar  \
Payment Currency                                                               
Australian Dollar                 127        0            0                0   
Bitcoin                             0       56            0                0   
Brazil Real                         0        0           57                0   
Canadian Dollar                     0        0            0              128   
Euro                                0        0            0                0   
Mexican Peso                        0        0            0                0   
Ruble                               0        0            0                0   
Rupee                               0        0            0                0   
Saudi Riyal                         0        0            0                0   
Shekel                              0        0            0                0   
Swiss Franc                         0   

In [4]:
# Save the currency-consistency evidence to outputs/tables/,
# so the finding is a recorded artefact rather than a screenshot.

OUT_TABLES = Path("..").resolve() / "outputs" / "tables"
OUT_TABLES.mkdir(parents=True, exist_ok=True)

# Full illicit currency table (all counts on the diagonal)
ccy_table.to_csv(OUT_TABLES / "ccy_table.csv")

# Headline numbers
summary = pd.DataFrame({
    "check": ["illicit same-currency", "illicit identical amounts",
              "legitimate same-currency", "illicit cross-currency rows",
              "legitimate cross-currency rows"],
    "value": ["100.00%", "100.00%", "98.58%", 0, 72170],
})
summary.to_csv(OUT_TABLES / "ccy_summary.csv", index=False)

print("Saved ccy_table.csv and ccy_summary.csv to outputs/tables/")

Saved ccy_table.csv and ccy_summary.csv to outputs/tables/
